# Atlas-Free ALE3DCNN Frozen Text Projection Variant

This Colab notebook tests whether reconstruction pretraining helps the atlas-free ALE3DCNN semantic/retrieval model.

Stage 1 trains `ALE volume -> ALE3DCNNEncoder -> 384 latent -> ALE3DCNNDecoder -> reconstructed ALE volume` using the same 4 mm atlas-free ALE cache settings as the current CNN run.

Stage 2 initializes the contrastive CNN brain encoder from the autoencoder and runs frozen/fine-tuned text projection variants, then runs the standard semantic evaluation suite.

## 1. Runtime, Dependencies, Repo

In [ ]:
import os, platform, subprocess, sys, time, json, shutil, traceback
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)

# Core dense-CNN path dependencies. PyG is not needed here.
!pip -q install nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn

REPO_URL = os.environ.get('NEUROVLM_REPO_URL', 'https://github.com/neurovlm/neurovlm.git')
REPO_BRANCH = os.environ.get('NEUROVLM_REPO_BRANCH', 'neurovlm_gnn')
REPO_DIR = os.environ.get('NEUROVLM_REPO_DIR', '/content/neurovlm_gnn')
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))
sys.path.insert(0, str(Path(REPO_DIR)))
print('Working directory:', os.getcwd())

## 2. Drive and Experiment Config

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

MODEL_NAME = 'ale_3dcnn_frozen_text_variant'
DRIVE_ROOT = '/content/drive/MyDrive/neurovlm'
DRIVE_DATA_DIR = f'{DRIVE_ROOT}/data_ale_3dcnn'
DRIVE_CACHE_DIR = f'{DRIVE_DATA_DIR}/ale_caches'
LOCAL_CACHE_DIR = '/content/ale_caches_ale_3dcnn'
RUNS_DIR = f'{DRIVE_ROOT}/runs_{MODEL_NAME}'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
COMPARISON_FILE = f'{RUNS_DIR}/frozen_text_variant_comparison.csv'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')

# Match the current atlas-free ALE3DCNN preprocessing.
RESOLUTION_MM = 4.0
KERNEL_FWHM_MM = 9.0
CROP_TO_BRAIN = True
NORMALIZE = 'max'
CLAMP = True
CACHE_DTYPE = 'float16'

# Model and training knobs.
LATENT_DIM = 384
BASE_CHANNELS = 48        # serious default; rerun with 64 after 48 if memory allows
NUM_BLOCKS = 4
DROPOUT = 0.1
NORM = 'group'
POOLING = 'max'
SEED = 42
AE_BATCH_SIZE = 64        # reconstruction pretraining batch size
NUM_WORKERS = 4
AE_EPOCHS = 100
AE_LR = 3e-4
AE_WEIGHT_DECAY = 1e-4
AE_LOSS = 'mse'  # 'mse' for continuous normalized ALE maps, or 'bce_logits'
CONTRASTIVE_EPOCHS = 150
CONTRASTIVE_BATCH_SIZE = 512
CONTRASTIVE_BATCH_SIZE_CANDIDATES = [1024, 512, 384, 256, 128]
LR_CNN = 1e-4
LR_PROJ = 1e-5
TEMPERATURE = 0.07
VAL_INTERVAL = 5
EARLY_STOPPING_PATIENCE = 25
TRAIN_SANITY_N = 512

for d in [DRIVE_ROOT, DRIVE_DATA_DIR, DRIVE_CACHE_DIR, LOCAL_CACHE_DIR, RUNS_DIR, EVAL_RESOURCE_DIR]:
    os.makedirs(d, exist_ok=True)

CACHE_BASENAME = f'atlas_free_ale_{int(RESOLUTION_MM)}mm_fwhm{str(KERNEL_FWHM_MM).replace(".", "p")}_crop_{CACHE_DTYPE}.pt'
ATLAS_FREE_CACHE = f'{LOCAL_CACHE_DIR}/{CACHE_BASENAME}'
DRIVE_ATLAS_FREE_CACHE = f'{DRIVE_CACHE_DIR}/{CACHE_BASENAME}'
LEGACY_DRIVE_CACHE_CANDIDATES = [
    DRIVE_ATLAS_FREE_CACHE,
    f'{DRIVE_CACHE_DIR}/atlas_free_{int(RESOLUTION_MM)}mm_fwhm{int(KERNEL_FWHM_MM)}_crop_{CACHE_DTYPE}.pt',
    f'{DRIVE_CACHE_DIR}/atlas_free_{RESOLUTION_MM:g}mm_fwhm{KERNEL_FWHM_MM:g}_crop_{CACHE_DTYPE}.pt',
]
for candidate in LEGACY_DRIVE_CACHE_CANDIDATES:
    if os.path.exists(candidate) and not os.path.exists(ATLAS_FREE_CACHE):
        shutil.copy2(candidate, ATLAS_FREE_CACHE)
        print('Copied Drive cache to local cache:', candidate, '->', ATLAS_FREE_CACHE)
        break

CONFIG = {k: v for k, v in globals().items() if k.isupper() and k not in {'CONFIG'}}
Path(RUNS_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(RUNS_DIR) / f'colab_config_{RUN_STAMP}.json', 'w') as f:
    json.dump({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, f, indent=2)
print(json.dumps({k: CONFIG[k] for k in ['RESOLUTION_MM','KERNEL_FWHM_MM','LATENT_DIM','BASE_CHANNELS','NUM_BLOCKS','AE_LOSS','AE_EPOCHS','CONTRASTIVE_EPOCHS','CONTRASTIVE_BATCH_SIZE','CONTRASTIVE_BATCH_SIZE_CANDIDATES']}, indent=2))
print('Runs:', RUNS_DIR)
print('Local cache:', ATLAS_FREE_CACHE)
print('Drive cache:', DRIVE_ATLAS_FREE_CACHE)

## 3. Build Atlas-Free ALE Dataset

In [ ]:
import argparse, importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader

from neurovlm.gnn.ale_cnn import ALE3DCNNAutoEncoder, ALE3DCNNEncoder, count_parameters
from neurovlm.gnn.ale_dataset import ALEPreprocessConfig, ALEVolumeDataset, build_or_load_ale_cache
from neurovlm.gnn.model import TextProjHead
from neurovlm.loss import InfoNCELoss

TRAIN_MODULE_PATH = str(Path(REPO_DIR) / 'experiments' / 'train_ale_cnn.py')
_spec = importlib.util.spec_from_file_location('train_ale_cnn_colab', TRAIN_MODULE_PATH)
train_ale_cnn = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(train_ale_cnn)

device = train_ale_cnn.which_device('cuda' if torch.cuda.is_available() else 'auto')
print('Device:', device)
torch.manual_seed(SEED)
np.random.seed(SEED)

dataset_args = argparse.Namespace(
    mode='atlas_free', kernel_fwhm_mm=KERNEL_FWHM_MM, resolution_mm=RESOLUTION_MM,
    crop_to_brain=CROP_TO_BRAIN, normalize=NORMALIZE, no_clamp=not CLAMP,
    cache_dtype=CACHE_DTYPE, cache_file=ATLAS_FREE_CACHE, force_rebuild_cache=False,
    max_papers=None, run_dir=str(Path(RUNS_DIR) / f'dataset_split_{RUN_STAMP}'),
    seed=SEED, val_frac=0.1, test_frac=0.1,
)
ds, train_ds, val_ds, test_ds, cache_payload, preprocess_config = train_ale_cnn.build_dataset(dataset_args)
print('Input shape:', ds.input_shape)
print('Splits:', len(train_ds), len(val_ds), len(test_ds))
if os.path.exists(ATLAS_FREE_CACHE) and not os.path.exists(DRIVE_ATLAS_FREE_CACHE):
    shutil.copy2(ATLAS_FREE_CACHE, DRIVE_ATLAS_FREE_CACHE)
    print('Copied newly built cache to Drive:', DRIVE_ATLAS_FREE_CACHE)

## 4. Stage 1 - Autoencoder Trainer

In [ ]:
def make_volume_loader(split_ds, batch_size, shuffle):
    return DataLoader(
        split_ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

def reconstruction_loss(logits, target, loss_name):
    target = target.float()
    if loss_name == 'mse':
        return F.mse_loss(logits, target)
    if loss_name == 'bce_logits':
        return F.binary_cross_entropy_with_logits(logits, target.clamp(0, 1))
    raise ValueError(loss_name)

def display_reconstruction(x, logits, loss_name):
    return torch.sigmoid(logits) if loss_name == 'bce_logits' else logits.clamp(0, 1)

@torch.no_grad()
def eval_autoencoder(model, loader, device):
    model.eval()
    losses = []
    for batch in loader:
        x = batch['volume'].float().to(device, non_blocking=True)
        logits = model(x)
        losses.append(float(reconstruction_loss(logits, x, AE_LOSS).detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def save_reconstruction_plot(model, split_ds, out_dir, device, n=4):
    import matplotlib.pyplot as plt
    model.eval()
    batch = next(iter(make_volume_loader(split_ds, n, shuffle=False)))
    x = batch['volume'].float().to(device)
    recon = display_reconstruction(x, model(x), AE_LOSS).detach().cpu()
    x = x.detach().cpu()
    n = min(n, x.shape[0])
    z = x.shape[-1] // 2
    fig, axes = plt.subplots(3, n, figsize=(3 * n, 8), squeeze=False)
    for i in range(n):
        true_slice = x[i, 0, :, :, z]
        recon_slice = recon[i, 0, :, :, z]
        diff = (true_slice - recon_slice).abs()
        for row, arr, title in [(0, true_slice, 'true'), (1, recon_slice, 'recon'), (2, diff, 'abs diff')]:
            ax = axes[row, i]
            ax.imshow(arr.T, origin='lower', cmap='magma' if row < 2 else 'viridis')
            ax.set_title(f'{title} {i}')
            ax.axis('off')
    fig.tight_layout()
    out_path = Path(out_dir) / 'reconstruction_examples.png'
    fig.savefig(out_path, dpi=160)
    plt.close(fig)
    return str(out_path)

def train_autoencoder():
    out_dir = Path(RUNS_DIR) / f'autoencoder_atlas_free_{RUN_STAMP}'
    ckpt_dir = out_dir / 'checkpoints'
    plot_dir = out_dir / 'plots'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    config = {
        'stage': 'cnn_autoencoder_pretraining', 'mode': 'atlas_free', 'input_shape': ds.input_shape,
        'latent_dim': LATENT_DIM, 'base_channels': BASE_CHANNELS, 'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT, 'norm': NORM, 'pooling': POOLING,
        'loss': AE_LOSS, 'epochs': AE_EPOCHS, 'batch_size': AE_BATCH_SIZE,
        'lr': AE_LR, 'weight_decay': AE_WEIGHT_DECAY,
        'preprocess_config': cache_payload['config'], 'cache_metadata': cache_payload['metadata'],
    }
    with open(out_dir / 'autoencoder_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    model = ALE3DCNNAutoEncoder(
        output_shape=ds.input_shape, base_channels=BASE_CHANNELS, num_blocks=NUM_BLOCKS,
        latent_dim=LATENT_DIM, dropout=DROPOUT, norm=NORM, pooling=POOLING,
    ).to(device)
    print('Autoencoder params:', count_parameters(model))
    optimizer = torch.optim.AdamW(model.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    train_loader = make_volume_loader(train_ds, AE_BATCH_SIZE, shuffle=True)
    val_loader = make_volume_loader(val_ds, AE_BATCH_SIZE, shuffle=False)
    history = {'train_loss': [], 'val_loss': [], 'epoch_time_sec': [], 'lr': []}
    best_loss = float('inf')
    best_state = None

    for epoch in range(AE_EPOCHS):
        start = time.perf_counter()
        model.train()
        losses = []
        for batch in train_loader:
            x = batch['volume'].float().to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                logits = model(x)
                loss = reconstruction_loss(logits, x, AE_LOSS)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        val_loss = eval_autoencoder(model, val_loader, device)
        train_loss = float(np.mean(losses))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['epoch_time_sec'].append(float(time.perf_counter() - start))
        history['lr'].append(float(optimizer.param_groups[0]['lr']))
        print(f'Epoch {epoch:03d}/{AE_EPOCHS} train={train_loss:.6f} val={val_loss:.6f}')
        state = {
            'encoder': model.encoder.state_dict(), 'decoder': model.decoder.state_dict(),
            'model': model.state_dict(), 'epoch': epoch, 'val_loss': val_loss,
            'history': history, 'config': config,
        }
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = state
            torch.save(state, ckpt_dir / 'best_cnn_autoencoder.pt')
            save_reconstruction_plot(model, val_ds, plot_dir, device)

    torch.save(state, ckpt_dir / 'last_cnn_autoencoder.pt')
    with open(out_dir / 'autoencoder_training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    with open(out_dir / 'autoencoder_run_info.json', 'w') as f:
        json.dump({'run_dir': str(out_dir), 'best_checkpoint': str(ckpt_dir / 'best_cnn_autoencoder.pt'), 'last_checkpoint': str(ckpt_dir / 'last_cnn_autoencoder.pt'), 'best_val_loss': best_loss}, f, indent=2)
    print('Autoencoder artifacts:', out_dir)
    return model, str(out_dir), str(ckpt_dir / 'best_cnn_autoencoder.pt')

## 5. Run Stage 1

In [ ]:
autoencoder, AUTOENCODER_RUN_DIR, BEST_AE_CHECKPOINT = train_autoencoder()
BEST_AE_CHECKPOINT

## 5b. Autoencoder Reconstruction Metrics

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

def resolve_autoencoder_checkpoint(checkpoint_path=None, run_dir=None):
    """Find an already-trained autoencoder checkpoint on Drive/local runs."""
    candidates = []
    if checkpoint_path:
        candidates.append(Path(checkpoint_path))
    if run_dir:
        candidates.append(Path(run_dir) / 'checkpoints' / 'best_cnn_autoencoder.pt')
    for root in [Path(RUNS_DIR), Path('/content/drive/MyDrive/neurovlm/runs_ale_3dcnn_autoencoder_pretrain')]:
        if root.exists():
            candidates.extend(sorted(root.glob('autoencoder_atlas_free_*/checkpoints/best_cnn_autoencoder.pt')))

    existing = [p for p in candidates if p.exists()]
    if not existing:
        raise FileNotFoundError(
            'Could not find best_cnn_autoencoder.pt. Set BEST_AE_CHECKPOINT or AUTOENCODER_RUN_DIR, '
            'or run Stage 1 first.'
        )
    best_path = max(existing, key=lambda p: p.stat().st_mtime)
    best_run_dir = best_path.parents[1]
    print('Using autoencoder checkpoint:', best_path)
    print('Autoencoder run directory:', best_run_dir)
    return str(best_path), str(best_run_dir)

def load_best_autoencoder(checkpoint_path):
    state = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg = state.get('config', {})
    input_shape = tuple(cfg.get('input_shape', ds.input_shape))
    model = ALE3DCNNAutoEncoder(
        output_shape=input_shape,
        base_channels=int(cfg.get('base_channels', BASE_CHANNELS)),
        num_blocks=int(cfg.get('num_blocks', NUM_BLOCKS)),
        latent_dim=int(cfg.get('latent_dim', LATENT_DIM)),
        dropout=float(cfg.get('dropout', DROPOUT)),
        norm=cfg.get('norm', NORM),
        pooling=cfg.get('pooling', POOLING),
    ).to(device)
    model.load_state_dict(state['model'])
    return model.eval()

@torch.no_grad()
def autoencoder_reconstruction_metrics(
    model,
    split_ds,
    *,
    split_name,
    batch_size=None,
    target_threshold=1e-6,
    top_frac=0.01,
):
    loader = make_volume_loader(split_ds, batch_size or AE_BATCH_SIZE, shuffle=False)
    mse_sum = 0.0
    mae_sum = 0.0
    voxel_count = 0
    map_pearsons = []
    voxel_aurocs = []
    voxel_average_precisions = []
    top_dices = []

    for batch in loader:
        x = batch['volume'].float().to(device, non_blocking=True)
        logits = model(x)
        recon = display_reconstruction(x, logits, AE_LOSS)
        diff = recon - x
        mse_sum += float(diff.pow(2).sum().detach().cpu())
        mae_sum += float(diff.abs().sum().detach().cpu())
        voxel_count += int(x.numel())

        scores = recon.detach().cpu().flatten(1).numpy()
        targets = x.detach().cpu().flatten(1).numpy()
        n_vox = targets.shape[1]
        k = max(1, int(round(top_frac * n_vox)))

        for y, s in zip(targets, scores):
            y_centered = y - y.mean()
            s_centered = s - s.mean()
            denom = float(np.linalg.norm(y_centered) * np.linalg.norm(s_centered))
            if denom > 0:
                map_pearsons.append(float(np.dot(y_centered, s_centered) / denom))

            labels = y > target_threshold
            if labels.any() and (~labels).any():
                voxel_aurocs.append(float(roc_auc_score(labels, s)))
                voxel_average_precisions.append(float(average_precision_score(labels, s)))

            true_top = np.argpartition(y, -k)[-k:]
            pred_top = np.argpartition(s, -k)[-k:]
            overlap = len(set(true_top.tolist()).intersection(pred_top.tolist()))
            top_dices.append(float(2 * overlap / (len(true_top) + len(pred_top))))

    return {
        'split': split_name,
        'n_maps': int(len(split_ds)),
        'reconstruction_mse': float(mse_sum / max(1, voxel_count)),
        'reconstruction_mae': float(mae_sum / max(1, voxel_count)),
        'voxel_auroc': float(np.mean(voxel_aurocs)) if voxel_aurocs else None,
        'voxel_average_precision': float(np.mean(voxel_average_precisions)) if voxel_average_precisions else None,
        'top_1pct_dice': float(np.mean(top_dices)) if top_dices else None,
        'map_pearson': float(np.mean(map_pearsons)) if map_pearsons else None,
        'target_threshold': float(target_threshold),
        'top_frac': float(top_frac),
    }

BEST_AE_CHECKPOINT, AUTOENCODER_RUN_DIR = resolve_autoencoder_checkpoint(
    globals().get('BEST_AE_CHECKPOINT'),
    globals().get('AUTOENCODER_RUN_DIR'),
)
best_autoencoder = load_best_autoencoder(BEST_AE_CHECKPOINT)
autoencoder_metric_rows = [
    autoencoder_reconstruction_metrics(best_autoencoder, val_ds, split_name='val'),
    autoencoder_reconstruction_metrics(best_autoencoder, test_ds, split_name='test'),
]
autoencoder_metrics = {
    'model': 'atlas_free_ale3dcnn_autoencoder',
    'checkpoint': BEST_AE_CHECKPOINT,
    'metrics': autoencoder_metric_rows,
}
metrics_path = Path(AUTOENCODER_RUN_DIR) / 'autoencoder_reconstruction_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(autoencoder_metrics, f, indent=2)
pd.DataFrame(autoencoder_metric_rows).to_csv(Path(AUTOENCODER_RUN_DIR) / 'autoencoder_reconstruction_metrics.csv', index=False)
print('Saved:', metrics_path)
pd.DataFrame(autoencoder_metric_rows)

## 6. Stage 2 - Contrastive Variants

In [ ]:
class FrozenAwareALETrainer(train_ale_cnn.ALETrainer):
    def __init__(self, brain_encoder, text_proj, args, device, freeze_brain_encoder=False):
        self.freeze_brain_encoder = freeze_brain_encoder
        if freeze_brain_encoder:
            for p in brain_encoder.parameters():
                p.requires_grad_(False)
        super().__init__(brain_encoder, text_proj, args, device)

    def _forward(self, batch):
        if self.freeze_brain_encoder:
            self.brain_encoder.eval()
        return super()._forward(batch)

def make_text_projector(init, freeze):
    if init == 'pretrained_infonce':
        import copy
        from neurovlm.models import ProjHead
        # ProjHead.from_pretrained is backed by an lru_cache, so deepcopy to
        # prevent one run from freezing or fine-tuning the module reused by
        # later variants in this same notebook runtime.
        text_proj = copy.deepcopy(ProjHead.from_pretrained('text_infonce'))
    elif init == 'random':
        text_proj = TextProjHead(in_dim=768, hidden_dim=512, out_dim=LATENT_DIM)
    else:
        raise ValueError(init)
    for p in text_proj.parameters():
        p.requires_grad_(not freeze)
    return text_proj

def build_brain_encoder_from_autoencoder(load_pretrained=True):
    encoder = ALE3DCNNEncoder(
        base_channels=BASE_CHANNELS, num_blocks=NUM_BLOCKS, out_dim=LATENT_DIM,
        dropout=DROPOUT, norm=NORM, pooling=POOLING,
    )
    if load_pretrained:
        state = torch.load(BEST_AE_CHECKPOINT, map_location='cpu', weights_only=False)
        encoder.load_state_dict(state['encoder'])
    return encoder

def preflight_contrastive_batch_size(brain_encoder, text_proj, freeze_brain=False):
    """Pick the largest actual InfoNCE batch that fits a forward/backward step."""
    candidates = sorted(set(CONTRASTIVE_BATCH_SIZE_CANDIDATES + [CONTRASTIVE_BATCH_SIZE]), reverse=True)
    loss_fn = InfoNCELoss(TEMPERATURE)
    trainable_params = [p for p in list(brain_encoder.parameters()) + list(text_proj.parameters()) if p.requires_grad]
    if not trainable_params:
        raise ValueError('No trainable parameters for this contrastive variant.')
    for bs in candidates:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            loader = DataLoader(train_ds, batch_size=min(bs, len(train_ds)), shuffle=False, num_workers=0)
            batch = next(iter(loader))
            x = batch['volume'].float().to(device)
            text = batch['text'].float().to(device)
            brain_encoder = brain_encoder.to(device)
            text_proj = text_proj.to(device)
            brain_encoder.train(not freeze_brain)
            text_proj.train(any(p.requires_grad for p in text_proj.parameters()))
            for p in trainable_params:
                p.grad = None
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                brain = brain_encoder(x)
                text_emb = text_proj(text)
                loss = loss_fn(brain, text_emb)
            loss.backward()
            peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
            for p in trainable_params:
                p.grad = None
            del batch, x, text, brain, text_emb, loss, loader
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f'Preflight selected actual contrastive batch_size={bs} peak_vram={peak_mb:.0f}MB')
            return int(bs), float(peak_mb)
        except RuntimeError as exc:
            message = str(exc).lower()
            if 'out of memory' not in message and 'mps backend out of memory' not in message:
                raise
            print(f'Preflight batch_size={bs} OOM; trying smaller actual batch.')
            for p in trainable_params:
                p.grad = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise RuntimeError(f'No contrastive batch size fit from candidates: {candidates}')

def contrastive_args_for(run_dir, variant, text_init, freeze_text):
    return argparse.Namespace(
        mode='atlas_free', model='ale_3dcnn', epochs=CONTRASTIVE_EPOCHS,
        batch_size=CONTRASTIVE_BATCH_SIZE, batch_size_auto=False, grad_accum_steps=1,
        lr_cnn=LR_CNN, lr_proj=LR_PROJ, warmup_epochs=5, temperature=TEMPERATURE,
        val_interval=VAL_INTERVAL, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        monitor_metric='paper_recall_curve_auc', base_channels=BASE_CHANNELS,
        num_blocks=NUM_BLOCKS, out_dim=LATENT_DIM, dropout=DROPOUT, norm=NORM,
        pooling=POOLING, mlp_hidden_dim=1024, kernel_fwhm_mm=KERNEL_FWHM_MM,
        resolution_mm=RESOLUTION_MM, crop_to_brain=CROP_TO_BRAIN, normalize=NORMALIZE,
        no_clamp=not CLAMP, cache_dtype=CACHE_DTYPE, cache_file=ATLAS_FREE_CACHE,
        force_rebuild_cache=False, build_cache_only=False, max_papers=None,
        text_proj_init=text_init, freeze_text_proj=freeze_text,
        device='cuda' if torch.cuda.is_available() else 'auto', amp=True,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), seed=SEED,
        val_frac=0.1, test_frac=0.1, checkpoint_dir=str(Path(run_dir) / 'checkpoints'),
        run_dir=str(run_dir), comparison_file=COMPARISON_FILE, save_plots=True,
        umap=True, eval_neurovlm_baseline=False, semantic_eval=True,
        eval_resource_dir=EVAL_RESOURCE_DIR, mesh_json=None, train_sanity_n=TRAIN_SANITY_N,
        variant=variant,
    )

def finalize_contrastive_run(trainer, args, variant_name, load_pretrained, freeze_brain, text_init, freeze_text):
    run_dir = Path(args.run_dir)
    all_metrics, eval_payloads = {}, {}
    if args.train_sanity_n and args.train_sanity_n > 0:
        train_metrics, train_brain, train_text = train_ale_cnn.evaluate_train_subset_sanity(trainer, train_ds, args.train_sanity_n)
        all_metrics['train_subset'] = train_metrics
        eval_payloads['train_subset'] = (train_metrics, train_brain, train_text)
    for split_name, split_ds in [('val', val_ds), ('test', test_ds)]:
        metrics, brain_emb, text_emb = trainer.evaluate(split_ds)
        all_metrics[split_name] = metrics
        eval_payloads[split_name] = (metrics, brain_emb, text_emb)
        print(f"{variant_name} {split_name}: AUC={metrics['paper_recall_curve_auc']:.4f} r@10={metrics['mean_recall@10']:.4f} MRR={metrics['mean_mrr']:.4f}")
    with open(run_dir / 'training_history.json', 'w') as f:
        json.dump(trainer.history, f, indent=2)
    with open(run_dir / 'eval_results.json', 'w') as f:
        json.dump(all_metrics, f, indent=2)
    for split_name, (_, brain_emb, text_emb) in eval_payloads.items():
        train_ale_cnn.recall_curve_frame(text_emb, brain_emb).to_csv(run_dir / f'{split_name}_recall_curve.csv', index=False)
        with open(run_dir / f'{split_name}_recall_curve.json', 'w') as f:
            json.dump(train_ale_cnn.recall_curve_payload(text_emb, brain_emb), f, indent=2)
    test_metrics, test_brain, test_text = eval_payloads['test']
    cov = test_ds.covariate_frame()
    diag_df = train_ale_cnn.retrieval_diagnostics(test_text, test_brain, cov)
    diag_df.to_csv(run_dir / 'test_retrieval_diagnostics.csv', index=False)
    train_ale_cnn.save_embedding_correlations(run_dir, test_brain, cov)
    train_ale_cnn.save_plots(run_dir, trainer.history, pd.read_csv(run_dir / 'test_recall_curve.csv'), diag_df, test_brain)

    from experiments.semantic_model_eval import run_ale_network_labeling, run_embedding_semantic_evaluations
    from neurovlm.data import load_masker
    semantic_summary = run_embedding_semantic_evaluations(
        model_name=variant_name, brain_embeddings=test_brain, text_embeddings=test_text,
        raw_text_embeddings=test_ds.raw_text_embeddings, pmids=test_ds.pmids,
        text_projector=trainer.text_proj, out_dir=run_dir, device=device,
        resource_dir=EVAL_RESOURCE_DIR, mesh_json=None,
        resource_use={
            'network_label_csv': 'networks_labels/network_test_set_labels.csv',
            'network_term_corpus_csv': 'networks_labels/network_terms_with_definitions.csv',
            'pmid_mesh_json': 'mesh_kg/mesh_annotations.json',
            'mesh_node_types': 'mesh_kg/mesh_kg_nodes.parquet',
        },
        extra_summary={
            'variant': variant_name, 'encoder_pretrained': load_pretrained,
            'brain_encoder_frozen': freeze_brain, 'text_proj_init': text_init,
            'text_proj_frozen': freeze_text, 'params': count_parameters(trainer.brain_encoder),
            'actual_batch_size': int(args.batch_size),
            'preflight_peak_vram_mb': float(getattr(args, 'preflight_peak_vram_mb', 0.0)),
            'peak_vram_mb': float(max(trainer.history['peak_vram_mb'] or [0.0])),
            'training_time_per_epoch': float(np.mean(trainer.history['epoch_time_sec'])),
            'preprocessing_type': 'atlas_free',
        },
    )
    network_metrics = run_ale_network_labeling(
        trainer=trainer, preprocess_config=preprocess_config, masker=load_masker(),
        out_dir=run_dir, device=device, resource_dir=EVAL_RESOURCE_DIR,
    )
    semantic_summary.update({
        'network_accuracy': network_metrics.get('accuracy'),
        'network_top_2_accuracy': network_metrics.get('top_2_accuracy'),
        'network_macro_auc': network_metrics.get('macro_auc'),
    })
    semantic_summary.update({k: v for k, v in network_metrics.items() if k.startswith('network_term_')})
    with open(run_dir / 'main_comparison_summary_row.json', 'w') as f:
        json.dump(semantic_summary, f, indent=2)
    pd.DataFrame([semantic_summary]).to_csv(run_dir / 'main_comparison_summary_row.csv', index=False)
    return {
        'variant': variant_name,
        'run_dir': str(run_dir),
        'summary': str(run_dir / 'main_comparison_summary_row.csv'),
        'actual_batch_size': int(args.batch_size),
        'preflight_peak_vram_mb': float(getattr(args, 'preflight_peak_vram_mb', 0.0)),
        'base_channels': int(BASE_CHANNELS),
        'num_blocks': int(NUM_BLOCKS),
        'text_proj_init': text_init,
        'freeze_text_proj': bool(freeze_text),
        'freeze_brain_encoder': bool(freeze_brain),
    }

def run_contrastive_variant(variant_name, *, load_pretrained, freeze_brain, text_init, freeze_text):
    run_dir = Path(RUNS_DIR) / f'{variant_name}_{RUN_STAMP}'
    run_dir.mkdir(parents=True, exist_ok=True)
    brain_encoder = build_brain_encoder_from_autoencoder(load_pretrained=load_pretrained)
    text_proj = make_text_projector(text_init, freeze_text)
    if freeze_brain:
        for p in brain_encoder.parameters():
            p.requires_grad_(False)
    actual_batch_size, preflight_peak_vram_mb = preflight_contrastive_batch_size(
        brain_encoder, text_proj, freeze_brain=freeze_brain
    )
    args = contrastive_args_for(run_dir, variant_name, text_init, freeze_text)
    args.batch_size = actual_batch_size
    args.preflight_peak_vram_mb = preflight_peak_vram_mb
    args.batch_size_candidates = CONTRASTIVE_BATCH_SIZE_CANDIDATES
    with open(run_dir / 'config.json', 'w') as f:
        json.dump(vars(args), f, indent=2)
    with open(run_dir / 'preprocessing_config.json', 'w') as f:
        json.dump({**cache_payload['config'], 'metadata': cache_payload['metadata']}, f, indent=2)
    trainer = FrozenAwareALETrainer(brain_encoder, text_proj, args, device, freeze_brain_encoder=freeze_brain)
    trainer.fit(train_ds, val_ds)
    trainer.restore_best()
    return finalize_contrastive_run(trainer, args, variant_name, load_pretrained, freeze_brain, text_init, freeze_text)

def run_or_load_contrastive_variant(variant_name, *, load_pretrained, freeze_brain, text_init, freeze_text):
    run_dir = Path(RUNS_DIR) / f'{variant_name}_{RUN_STAMP}'
    summary_path = run_dir / 'main_comparison_summary_row.csv'
    config_path = run_dir / 'config.json'
    if summary_path.exists():
        info = {
            'variant': variant_name,
            'run_dir': str(run_dir),
            'summary': str(summary_path),
            'base_channels': int(BASE_CHANNELS),
            'num_blocks': int(NUM_BLOCKS),
            'text_proj_init': text_init,
            'freeze_text_proj': bool(freeze_text),
            'freeze_brain_encoder': bool(freeze_brain),
            'loaded_existing': True,
        }
        if config_path.exists():
            cfg = json.loads(config_path.read_text())
            info['actual_batch_size'] = int(cfg.get('batch_size', 0))
            info['preflight_peak_vram_mb'] = float(cfg.get('preflight_peak_vram_mb', 0.0))
        print(f'Using existing completed run: {run_dir}')
        return info
    return run_contrastive_variant(
        variant_name,
        load_pretrained=load_pretrained,
        freeze_brain=freeze_brain,
        text_init=text_init,
        freeze_text=freeze_text,
    )

## 7. Run Contrastive Variants

Serious-run variants use `base_channels=48`, `num_blocks=4`, the largest actual InfoNCE batch selected from `[1024, 512, 384, 256, 128]`, and `text_proj_init='pretrained_infonce'`.

Variants:

1. scratch atlas-free ALE3DCNN + pretrained text projection frozen
2. scratch atlas-free ALE3DCNN + pretrained text projection trainable
3. frozen autoencoder-pretrained CNN encoder + pretrained text projection trainable
4. fine-tuned autoencoder-pretrained CNN encoder + pretrained text projection frozen
5. fine-tuned autoencoder-pretrained CNN encoder + pretrained text projection trainable

The frozen-CNN plus frozen-text case is skipped because it has no trainable contrastive parameters.

In [ ]:
RUN_INFOS = []

# Isolated promising side variant. The main notebook should stay focused on
# ae_pretrained_finetune_cnn_pretrained_text_trainable and its global-context
# descendants. This one freezes the pretrained text projection for later
# exploration.
RUN_INFOS.append(run_or_load_contrastive_variant(
    'ae_pretrained_finetune_cnn_pretrained_text_frozen',
    load_pretrained=True, freeze_brain=False, text_init='pretrained_infonce', freeze_text=True,
))

pd.DataFrame(RUN_INFOS)


## 8. Compare Important Metrics

In [ ]:
IMPORTANT_METRICS = [
    'model', 'variant', 'actual_batch_size', 'peak_vram_mb', 'training_time_per_epoch',
    'network_term_recall@5', 'network_term_recall@10', 'network_term_mrr',
    'mesh_recall@10', 'semantic_recall@10', 'semantic_recall@50',
    'semantic_paper_style_recall_curve_auc',
    'exact_pmid_paper_recall_curve_auc', 'exact_pmid_recall@10',
]

def load_summary(path):
    path = Path(path)
    if path.is_dir():
        path = path / 'main_comparison_summary_row.csv'
    if not path.exists():
        print('Missing summary:', path)
        return None
    return pd.read_csv(path).iloc[0].to_dict()

rows = []
for info in RUN_INFOS:
    row = load_summary(info['run_dir'])
    if row:
        row['variant'] = info['variant']
        rows.append(row)

# Existing pretrained NeuroVLM MLP semantic baseline, if available on this Drive/runtime.
BASELINE_CANDIDATES = [
    Path(REPO_DIR) / 'experiments/runs/neurovlm_pubmed_semantic_baseline',
    Path(DRIVE_ROOT) / 'runs/neurovlm_pubmed_semantic_baseline',
    Path('/content/drive/MyDrive/neurovlm/runs/neurovlm_pubmed_semantic_baseline'),
]
for p in BASELINE_CANDIDATES:
    row = load_summary(p)
    if row:
        row['variant'] = 'pretrained_neurovlm_mlp_baseline'
        rows.append(row)
        break

comparison = pd.DataFrame(rows)
available = [c for c in IMPORTANT_METRICS if c in comparison.columns]
comparison[available].sort_values([c for c in ['network_term_recall@10', 'mesh_recall@10'] if c in comparison.columns], ascending=False)

## 9. Interpretation Template

In [ ]:
out_path = Path(RUNS_DIR) / f'autoencoder_pretraining_interpretation_{RUN_STAMP}.md'
with open(out_path, 'w') as f:
    f.write('# ALE3DCNN Autoencoder Pretraining Interpretation\n\n')
    f.write('- If an autoencoder-pretrained CNN beats the scratch CNN, representation pretraining was likely a limiting factor.\n')
    f.write('- If autoencoder-pretrained CNN still does not beat the pretrained NeuroVLM MLP baseline, the original MLP/NeuroVLM latent representation may be stronger, or atlas-free ALE/training objective choices remain limiting.\n')
    f.write('- Treat exact PMID AUC as a strict diagnostic. Prioritize network term recall@5/10, network term MRR, MeSH recall@10, and semantic recall@10/50.\n')
print(out_path)